# 05 - Deploy the data agent

Copyright (c) Microsoft Corporation. Licensed under the MIT License.

Deploys **and publishes** the *Fabric Arch Review - Data Agent*. This is a standalone notebook - it is
**not** part of the pipeline. FOS setup publishes the required `fabric-data-agent-sdk==0.1.30a0` in **FAR_Runtime** and binds this notebook to it.

**Run order:** enable governance reporting and run the pipeline once, then run all cells of this notebook.
The agent grounds on Gold tables via the SDK, which lets the Fabric backend
*enumerate* each datasource's schema. Verify source permissions, table availability and publication;
deployment does not guarantee every source resolves.

**Central audience only:** the optional Workspace Owner model's RLS does not secure this agent,
its governance model or its Lakehouse sources. Do not share this agent or grant its source
permissions to owner consumers as part of owner-report approval.

**Runtime:** keep `DATA_AGENT_RUNTIME = "standard"` for production. To compare preview,
deploy a separately named test agent against the same sources and set its runtime to `"preview"`.
Publishing captures the runtime setting. Runtime selection does not change source permissions.


In [ ]:
DATA_AGENT_NAME = "Fabric Arch Review - Data Agent"
SEMANTIC_MODEL_NAME = "Fabric Arch Review - Governance"
DATA_AGENT_RUNTIME = "standard"
LAKEHOUSE_NAME = "FOS_Lakehouse"


In [ ]:
from importlib.metadata import version
if version("fabric-data-agent-sdk") != "0.1.30a0":
    raise RuntimeError("Rerun FOS_Install_FAR to publish the SDK in FAR_Runtime, then restart this notebook session with that Environment.")


In [ ]:
import os, sys, runpy
from pathlib import Path
import notebookutils
ctx = notebookutils.runtime.context
if (ctx.get("defaultLakehouseName") != "FOS_Lakehouse" or not ctx.get("defaultLakehouseId")
        or ctx.get("defaultLakehouseWorkspaceId") != (ctx.get("currentWorkspaceId") or ctx.get("workspaceId"))):
    raise ValueError("Attach this workspace's FOS_Lakehouse as default and restart the session.")
REPO_DIR = "/lakehouse/default/Files/far/runtime/source"
if not Path(REPO_DIR, "fabric", "bootstrap.py").is_file():
    raise FileNotFoundError("FAR runtime not staged. Deploy the architecture-review module before running.")
runpy.run_path(str(Path(REPO_DIR, "fabric", "bootstrap.py")))

with open(os.path.join(REPO_DIR, "VERSION"), encoding="utf-8-sig") as _vf:
    _far_ver = _vf.read().strip()

# ---- deploy + publish the agent (backend enumerates the schema -> no phantom tables) ----
from reports.agent.sdk_deploy import deploy_agent
deploy_agent(
    agent_name=DATA_AGENT_NAME,
    model_name=SEMANTIC_MODEL_NAME,
    lakehouse_name=LAKEHOUSE_NAME,
    version=_far_ver,
    runtime=DATA_AGENT_RUNTIME,
    publish=True,
)
from orchestration.fabric_api import FabricClient
from orchestration.folders import find_item, organize
import notebookutils
_folder_client = FabricClient(lambda: notebookutils.credentials.getToken('pbi'))
_ctx = notebookutils.runtime.context
_workspace = _ctx.get('currentWorkspaceId') or _ctx.get('workspaceId')
_agent_item = find_item(_folder_client, _workspace, DATA_AGENT_NAME, 'DataAgent')
if _agent_item is None:
    raise ValueError('Published central Agent could not be uniquely resolved for folder placement.')
organize(_folder_client, _workspace, [(_agent_item['id'], 'DataAgent', 'FAR/Governance'),
         (_ctx['currentNotebookId'], 'Notebook', 'FAR/Governance')])
print("Data agent deployed + published:", DATA_AGENT_NAME)
